<a href="https://colab.research.google.com/github/BrunoArias115/03MIAR---Algoritmos-de-Optimizacion/blob/main/Trabajo_Pr%C3%A1ctico_Algoritmos(V2%2Cno_borrar).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Trabajo Práctico<br>
Nombre y Apellidos: Bruno Arias Correa  <br>
Url: https://github.com/BrunoArias115/03MIAR---Algoritmos-de-Optimizacion.git<br>
Google Colab: https://colab.research.google.com/drive/1lW85Dl8kUpbdcxVAMeALzuGZe2n9xA89?usp=sharing <br>
Problema:

> Organizar los horarios de partidos de una jornada de La Liga<br>


Descripción del problema:

Desde la La Liga de fútbol profesional se pretende organizar los horarios de los
partidos de liga de cada jornada. Se conocen algunos datos que nos deben llevar a
diseñar un algoritmo que realice la asignación de los partidos a los horarios de forma
que maximice la audiencia.

• Los horarios disponibles se conocen a priori y son los siguientes:

Viernes 20

Sábado 12,16,18,20

Domingo 12,16,18,20

Lunes 20







                                        

#Modelo
- ¿Como represento el espacio de soluciones?
- ¿Cual es la función objetivo?
- ¿Como implemento las restricciones?

In [ ]:
## ¿Cómo represento el espacio de soluciones?

"""Una solución se modela como un vector de tamaño 10, donde cada posición representa un partido de la jornada y el valor almacenado corresponde al horario asignado.

S = (h1, h2, ..., h10)

donde:

- hi ∈ H
- H = {V20, S12, S16, S18, S20, D12, D16, D18, D20, L20}

Si se restringe a una asignación única de horarios, el espacio corresponde a las permutaciones de 10 elementos.

|S| = 10! = 3,628,800

El espacio crece factorialmente, por lo que no es abordable mediante fuerza bruta si el problema escala."""


## ¿Cuál es la función objetivo?

"""El objetivo es maximizar la audiencia total de la jornada.

La audiencia de cada partido depende de:

1. Categoría de los equipos
2. Coeficiente del horario asignado
3. Penalización por coincidencia de partidos simultáneos

Formalmente:

Max F(S) = Σ (Ai * C_horario(i) * P_coincidencia(i))

Donde:

- Ai = audiencia base del partido i
- C_horario(i) = coeficiente multiplicador según día/hora
- P_coincidencia(i) = penalización si existen partidos simultáneos

La función no es lineal debido a la interacción entre partidos que comparten horario."""


## ¿Cómo implemento las restricciones?

"""Las restricciones del problema son:

1. Debe existir exactamente:
   - 1 partido el viernes 20h
   - 1 partido el lunes 20h

2. Cada partido debe asignarse a un único horario válido.

3. Si se prohíben coincidencias, cada horario puede usarse una sola vez.

Implementación:

- Se generan únicamente soluciones válidas (permutaciones).
- Alternativamente, se pueden penalizar soluciones inválidas en la función objetivo mediante un costo muy alto."""


#Análisis
- ¿Que complejidad tiene el problema?. Orden de complejidad y Contabilizar el espacio de soluciones

In [ ]:
## ¿Qué complejidad tiene el problema?

"""Si modelamos como asignación única:

O(n!) con n = 10

10! = 3,628,800

Si permitimos coincidencias:

O(n^n)"""

## Contabilización del espacio de soluciones

"""Caso sin coincidencias:

|S| = 10!

Caso con coincidencias:

|S| = 10^10

El crecimiento es exponencial/factorial, lo que justifica el uso de metaheurísticas."""


#Diseño
- ¿Que técnica utilizo? ¿Por qué?

In [ ]:
"""Se utiliza una metaheurística de Búsqueda Local (Hill Climbing).

Justificación:

- El espacio de búsqueda es grande y combinatorio.
- La función objetivo presenta dependencias entre variables (penalización por coincidencia).
- No es viable evaluar todas las combinaciones.
- La búsqueda local permite mejorar iterativamente una solución inicial explorando su vecindario.

Ventajas:

- Implementación sencilla
- Bajo costo computacional
- Mejora rápida sobre soluciones aleatorias

Limitación:

- Puede quedar atrapado en óptimos locales.

# **"CODIGO"**

In [1]:
import random
import math
from collections import Counter

In [2]:
# 3 equipos A, 11 B, 6 C
equipos = {
    "A1":"A","A2":"A","A3":"A",
    "B1":"B","B2":"B","B3":"B","B4":"B","B5":"B","B6":"B","B7":"B","B8":"B","B9":"B","B10":"B","B11":"B",
    "C1":"C","C2":"C","C3":"C","C4":"C","C5":"C","C6":"C"
}

In [3]:
def generar_partidos(equipos):
    lista = list(equipos.keys())
    random.shuffle(lista)
    partidos = []
    for i in range(0,20,2):
        partidos.append((lista[i], lista[i+1]))
    return partidos

partidos = generar_partidos(equipos)

In [4]:
audiencia_base = {
    ("A","A"):2.0,
    ("A","B"):1.3,
    ("A","C"):1.0,
    ("B","B"):0.9,
    ("B","C"):0.75,
    ("C","C"):0.47
}

In [5]:
horarios = [
    "V20",
    "S12","S16","S18","S20",
    "D12","D16","D18","D20",
    "L20"
]

In [6]:
coef_horario = {
    "V20":0.4,
    "S12":0.55,
    "S16":0.7,
    "S18":0.8,
    "S20":1.0,
    "D12":0.45,
    "D16":0.75,
    "D18":0.85,
    "D20":1.0,
    "L20":0.4
}

In [7]:
def obtener_audiencia_base(p):
    c1 = equipos[p[0]]
    c2 = equipos[p[1]]
    if (c1,c2) in audiencia_base:
        return audiencia_base[(c1,c2)]
    return audiencia_base[(c2,c1)]

In [8]:
def funcion_objetivo(solucion, partidos):

    total = 0

    conteo = Counter(solucion)

    for i in range(len(partidos)):
        base = obtener_audiencia_base(partidos[i])
        horario = solucion[i]
        coef = coef_horario[horario]

        # penalización coincidencias
        n = conteo[horario]
        if n == 1:
            penal = 1
        elif n == 2:
            penal = 0.9
        elif n == 3:
            penal = 0.8
        else:
            penal = 0.7

        total += base * coef * penal

    return total

In [9]:
def solucion_inicial():
    sol = horarios.copy()
    random.shuffle(sol)
    return sol

In [10]:
def vecino(sol):
    nueva = sol.copy()
    i,j = random.sample(range(len(sol)),2)
    nueva[i], nueva[j] = nueva[j], nueva[i]
    return nueva

In [11]:
def hill_climbing(partidos, iteraciones=5000):

    actual = solucion_inicial()
    mejor = actual
    mejor_valor = funcion_objetivo(actual, partidos)

    for _ in range(iteraciones):
        candidato = vecino(actual)
        valor = funcion_objetivo(candidato, partidos)

        if valor > mejor_valor:
            mejor = candidato
            mejor_valor = valor
            actual = candidato

    return mejor, mejor_valor

In [12]:
mejor_sol, mejor_valor = hill_climbing(partidos)

print("Mejor audiencia:", mejor_valor)
print("\nAsignación final:\n")

for i,p in enumerate(partidos):
    print(p, "->", mejor_sol[i])

Mejor audiencia: 7.035

Asignación final:

('C3', 'A1') -> S20
('B7', 'B11') -> D18
('C4', 'B6') -> V20
('B4', 'B9') -> S18
('C5', 'B1') -> S12
('C6', 'B3') -> L20
('A3', 'A2') -> D20
('B10', 'B5') -> D16
('C2', 'B2') -> S16
('C1', 'B8') -> D12
